# 7.25 怎樣判斷多花計算值得？


如果 medium 的新題正確率比 high 高，應該選哪個？先把需求說清楚：假設我們要處理大量有限訂單題，希望在平均計算上限內答對最多題。這時 high 這個名稱沒有優先權，真正重要的是留出題上的品質與成本。

先固定模型、提示、題目與解碼規則，再讓每個檔位回答相同題目。逐題保存答案、是否正確、是否遵守硬上限、實際工作量與耗時；比較服務時還要記版本與量測日期。輪數、token 數與秒數是不同單位，不能直接相加。多候選方法要把全部候選及挑選成本算進去。

下面是假設的四題最後測試紀錄，模型與規則已凍結，只教計算報表。`ok` 是依已知正解打分的0／1；`work` 暫以同模型執行的輪數作代理量，沒有宣稱一輪等於固定秒數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
logs = {
    "low": {"ok": [1, 1, 0, 0], "work": [1, 1, 1, 1]},
    "medium": {"ok": [1, 1, 1, 1], "work": [3, 3, 3, 3]},
    "high": {"ok": [1, 1, 1, 0], "work": [5, 5, 5, 5]},
}
for name, row in logs.items():
    print(name, "正確率", sum(row["ok"]) / 4, "平均工作", sum(row["work"]) / 4)

這張人工表的 medium 是4/4、high 是3/4，所以多算並未改善本表。它說明為何必須保留中間檔的固定基線；四題太少，也不能當某個模型的穩定排序。實際評估除了平均分，還看分任務表現、失敗例與足夠樣本下的延遲分布，避免容易題掩蓋難題。

**Auto** 是服務先根據輸入選預算。技術上可行，但多一個選擇器就多一份成本與犯錯機會：看似簡單的題可能有隱藏條件，分布改變也會使舊規則失準。選擇器不能看到真解；用真解回頭挑每題最好的檔位，只是離線理想上界。若所有檔都錯，那題在上界裡也仍然錯。

要試 Auto，就用 validation 選規則與門檻，凍結後再碰 test。若 high 只是允許更多工作，也應記它實際用了多少。對缺少必要資訊的題，先問清楚可能比加算更合理；第9章會把「缺資訊」做成明確判斷，第19章才把判斷器與解題器接起來。

練習：假設只允許平均3輪，low、medium、high 哪些符合上限？medium 在本表最好，但不能用這張 test 表再挑規則。接下來把[C.1–C.5](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html)當作必修：依序比較可見步驟、候選份數與驗證，理解工作量究竟花在哪裡。
